<a href="https://colab.research.google.com/github/ScrapMetal1/simple-proxy/blob/dev/notebooks/training_starter1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Watt The Hack Controller Training Starter (v0.1.0)

<a href="https://colab.research.google.com/github/AaronEliasZachariah/Watt-The-Hack-Engine-Public/blob/main/notebooks/training_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Welcome to the advanced track! In this notebook, you will learn how to write a controller for the Watt The Hack simulation engine. We will cover installing the engine, writing basic controllers, evaluating them against scenarios, and exporting an ML model's weights.

## 1. Install the Simulation Engine
First, let's install the Watt The Hack engine directly from GitHub.

In [101]:
!pip install git+https://github.com/AaronEliasZachariah/Watt-The-Hack-Engine-Public.git@main

  Cloning https://github.com/AaronEliasZachariah/Watt-The-Hack-Engine-Public.git (to revision main) to /tmp/pip-req-build-uqgqt9dn
  Running command git clone --filter=blob:none --quiet https://github.com/AaronEliasZachariah/Watt-The-Hack-Engine-Public.git /tmp/pip-req-build-uqgqt9dn
  Resolved https://github.com/AaronEliasZachariah/Watt-The-Hack-Engine-Public.git to commit a2d2e81d9721a0db5afa6b822d29f23c68e4701e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Setting up the Simulation
We can instantiate the `NetworkEngine` and run it for a few steps with a simple "do-nothing" controller.

In [102]:
from watt_the_hack.engine import Engine

engine = Engine()

# 1. Define a simple starting state
state = {
    "time": 0,
    "demand": 50.0,
    "solar": 30.0,
    "soc": 0.5,
    "profiles": {
        "demand": [50.0, 55.0, 60.0, 65.0, 70.0],
        "solar": [30.0, 25.0, 20.0, 15.0, 10.0],
    },
    "price_profile": [0.24, 0.24, 0.24, 0.24, 0.24],
    "price": 0.24,
}

# 2. Define a simple controller
def do_nothing_controller(state):
    # battery_flow_kw: positive = discharge, negative = charge
    return {"battery_flow_kw": 0.0, "emergency_generator": 0.0, "curtail_solar": 0.0}

# 3. Run the engine for a few steps
for _ in range(4):
    action = do_nothing_controller(state)
    state, outputs = engine.step(state, action)
    print(f"Step {state['time']} | Net Grid: {outputs['net_grid_power']:>5.1f} kW | Cost: ${outputs['step_cost']:.2f}")


Step 1 | Net Grid:  20.0 kW | Cost: $21.38
Step 2 | Net Grid:  30.0 kW | Cost: $12.16
Step 3 | Net Grid:  40.0 kW | Cost: $12.85
Step 4 | Net Grid:  50.0 kW | Cost: $13.54


## 3. A Rule-Based Controller
A slightly more advanced controller that charges when prices are low and discharges when prices are high.

In [103]:
def rule_based_controller(state):
    action = {
        "battery_flow_kw": 0.0,
        "emergency_generator": 0.0,
        "curtail_solar": 0.0,
        "fcas_reserve_kw": 0.0,
    }

    current_price = state["price"]
    soc = state["soc"]

    # Simple arbitrage logic
    if current_price < 0.10 and soc < 0.9:
        action["battery_flow_kw"] = -20.0  # Charge
    elif current_price > 0.30 and soc > 0.1:
        action["battery_flow_kw"] = 20.0   # Discharge

    # Hold some FCAS if we aren't doing anything else
    if action["battery_flow_kw"] == 0.0:
        action["fcas_reserve_kw"] = 10.0

    return action

state = {
    "time": 0,
    "demand": 50.0,
    "solar": 30.0,
    "soc": 0.5,
    "profiles": {
        "demand": [50.0] * 96,
        "solar": [30.0] * 96,
    },
    "price_profile": [0.24] * 96,
    "price": 0.24,
    "forecast_config": {"horizon_steps": 12}
}
total_cost = 0
for _ in range(96):
    action = rule_based_controller(state)
    state, outputs = engine.step(state, action)
    total_cost += outputs["step_cost"]

print(f"Rule-based total cost: ${total_cost:.2f}")


Rule-based total cost: $142.40


## 4. Reinforcement Learning Scaffold
You can wrap the engine in a gym environment. Here is a simple scaffold.

In [104]:
import numpy as np

class SimpleModel:
    """A dummy model to demonstrate weight extraction."""
    def __init__(self):
        # Random weights for illustration: 4 actions based on 3 state variables
        self.weights = np.random.randn(4, 3)

    def predict(self, obs):
        return np.dot(self.weights, obs)

model = SimpleModel()

## 5. Compiling Weights to Literals
Since the sandbox restricts imports (no `numpy` or `torch`), you must bake your model's weights directly into your submission code as Python literals.

In [105]:
def export_weights_to_code(model, filename="submission.py"):
    weights_list = model.weights.tolist()

    code = f"""# Auto-generated submission
WEIGHTS = {weights_list}

def controller(state):
    # Map state to vector
    obs = [state.get("soc", 0), state.get("price", 0), state.get("demand", 0)]

    # Manual dot product
    actions = [sum(w * o for w, o in zip(weight_row, obs)) for weight_row in WEIGHTS]

    # Action clamping and conversion
    return {{
        \"battery_flow_kw\": max(-50.0, min(50.0, actions[0])),
        \"emergency_generator\": max(0.0, min(50.0, actions[1])),
        \"curtail_solar\": max(0.0, min(state.get(\"solar\", 0.0), actions[2])),
        \"fcas_reserve_kw\": max(0.0, min(50.0, actions[3])),
    }}
"""
    with open(filename, "w") as f:
        f.write(code)
    print(f"Exported controller to {filename}")

export_weights_to_code(model)

# Let's view the generated code:
with open("submission.py", "r") as f:
    print(f.read())

Exported controller to submission.py
# Auto-generated submission
WEIGHTS = [[-1.540526984711403, -0.8218701522121057, -1.3967689098769656], [1.3015642413240056, -0.8795011013367334, -1.9583110672825248], [-0.5914675357146403, 0.4655359829396769, -0.0800889258868435], [0.1503723307017673, -0.04094883975198632, 1.3508026525300565]]

def controller(state):
    # Map state to vector
    obs = [state.get("soc", 0), state.get("price", 0), state.get("demand", 0)]
    
    # Manual dot product
    actions = [sum(w * o for w, o in zip(weight_row, obs)) for weight_row in WEIGHTS]
    
    # Action clamping and conversion
    return {
        "battery_flow_kw": max(-50.0, min(50.0, actions[0])),
        "emergency_generator": max(0.0, min(50.0, actions[1])),
        "curtail_solar": max(0.0, min(state.get("solar", 0.0), actions[2])),
        "fcas_reserve_kw": max(0.0, min(50.0, actions[3])),
    }



## 6. Submitting to the Cloud Leaderboard

When you're ready to score your controller against the hidden judging scenarios, submit it to the **cloud evaluation API**. This is different from the quick `/sim/run` playground — your code runs in an isolated Kubernetes container, gets a final score, and lands on the leaderboard.

**The flow is asynchronous:**
1. POST your ZIP to `/submissions` → get a `submission_id` back immediately
2. Poll `/submissions/{id}` until status is `COMPLETED`
3. GET `/submissions/{id}/score` to retrieve your scores

**Your ZIP must contain:**
- `strategy.py` — defines `class Strategy` with `.step(state)` (and optionally `.plan()` / `.replan()` for agentic scenarios)
- `requirements.txt` — your pip dependencies (`openai`, `anthropic` available for agentic scenarios)
- `metadata.json` — `{"entrypoint": "strategy.py", "class_name": "Strategy", "scenario_id": "duck_curve"}`

**You need:**
- `API_URL`: the gateway URL provided by organisers (e.g. `https://api.watt-the-hack.example.com`)
- `TEAM_TOKEN`: your team's `X-Team-Token` issued by organisers

In [117]:
%%writefile strategy.py
# Baseline conditional controller — WITH FCAS.
#
# Same skeleton as `baseline_conditional_controller.py`, plus opportunistic
# FCAS reservation on whatever inverter capacity isn't being used by the
# battery this step. FCAS pays $0.04/kW/h for held-available capacity —
# pure passive income when the battery is idle.
#
# Inverter constraint enforced by the engine:
#     |battery_flow_kw| + fcas_reserve_kw <= max_inverter_kw (50 kW)
#
# We reserve (50 - |battery_flow|) - FCAS_HEADROOM_KW, keeping a small
# safety margin so the battery can still react. FCAS is also dropped near
# SOC extremes — a battery sitting at 95% SOC cannot actually respond to a
# downward frequency event, so claiming FCAS there is dishonest.
#
# On scenarios with `features.fcas = false` (e.g. Duck Curve), the engine
# gate zeros out the FCAS field — this controller will then score
# identically to the no-FCAS version.


def controller(state, _hist=[]):
    PEAK_SHAVE_KW = 80.0
    WEAR_PER_KWH = 0.05
    SOC_MIN = 0.10
    SOC_MAX = 0.95
    INVERTER_KW = 50.0
    EXPORT_CAP_KW = 50.0
    IMPORT_CAP_KW = 120.0
    FCAS_HEADROOM_KW = 10.0  # battery flow headroom kept available

    soc = float(state.get("soc", 0.5))
    demand = float(state.get("demand", 0.0))
    solar = float(state.get("solar", 0.0))
    price = float(state.get("price", 0.0))

    budget = float(state.get("battery_throughput_remaining_kwh", 1e9))
    t = int(state.get("time", 0))
    steps_left = max(1, 96 - t)
    fair_share = budget / steps_left
    step_cap = min(INVERTER_KW, fair_share * 4.0 / 0.25)

    _hist.append(price)
    if len(_hist) > 32:
        del _hist[: len(_hist) - 32]
    sorted_h = sorted(_hist)
    n = len(sorted_h)
    cheap_p = sorted_h[max(0, n // 3 - 1)]
    exp_p = sorted_h[min(n - 1, (2 * n) // 3)]

    raw_net = demand - solar
    battery_flow = 0.0

    # Rule 1: soak overvoltage excess only
    if raw_net < -EXPORT_CAP_KW and soc < SOC_MAX and budget > 0.5:
        excess = -raw_net - EXPORT_CAP_KW
        soak = min(excess, step_cap, (SOC_MAX - soc) * 100.0 / 0.25)
        battery_flow = -soak

    # Rule 2: peak shave
    elif raw_net > PEAK_SHAVE_KW and soc > SOC_MIN and budget > 0.5:
        shave = min(
            raw_net - PEAK_SHAVE_KW,
            step_cap,
            (soc - SOC_MIN) * 100.0 / 0.25,
        )
        battery_flow = shave

    # Rule 3: arbitrage (wear-aware)
    elif (exp_p - cheap_p) > 2 * WEAR_PER_KWH and budget > 0.5:
        if price <= cheap_p and soc < SOC_MAX:
            cap = min(step_cap, (SOC_MAX - soc) * 100.0 / 0.25)
            battery_flow = -min(cap, max(0.0, IMPORT_CAP_KW - raw_net - 20.0))
        elif price >= exp_p and soc > SOC_MIN and raw_net > 0:
            cap = min(raw_net, step_cap, (soc - SOC_MIN) * 100.0 / 0.25)
            battery_flow = cap

    # FCAS reservation on otherwise-idle inverter capacity
    used = abs(battery_flow)
    fcas = max(0.0, INVERTER_KW - used - FCAS_HEADROOM_KW)
    # Don't claim FCAS the battery couldn't actually deliver from extreme SOC.
    if soc < SOC_MIN + 0.05 or soc > SOC_MAX - 0.05:
        fcas = 0.0

    # Safety nets
    net_after = demand - solar - battery_flow
    curtail = max(0.0, -net_after - EXPORT_CAP_KW)
    diesel = max(0.0, net_after - IMPORT_CAP_KW)

    return {
        "battery_flow_kw": battery_flow,
        "curtail_solar": curtail,
        "emergency_generator": diesel,
        "fcas_reserve_kw": fcas,
    }


Overwriting strategy.py


In [118]:
import io
import json
import time
import zipfile
import requests

# ── CONFIG — set these for your team ───────────────────────────────────────
API_URL    = "http://34.129.156.145"   # official admin server for judging
TEAM_ID    = "bdf471f6-20e1-42d7-8878-ae98f598551f"            # <- provided by organisers
TEAM_TOKEN = "ojffyNk726zc4DdkpXWiRQ"      # <- provided by organisers

# ── 1. Write your strategy (class-based for cloud submission) ─────────────


REQUIREMENTS = ""  # add e.g. "openai==1.35.0\n" if using LLM
METADATA = {"entrypoint": "strategy.py", "function_name": "controller",
    "scenario_id": "duck_curve", "strategy_name": "My Baseline"}

# ── 2. Package as ZIP in memory ───────────────────────────────────────────
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("strategy.py", arcname="strategy.py")
    zf.writestr("requirements.txt", REQUIREMENTS)
    zf.writestr("metadata.json",    json.dumps(METADATA))
buf.seek(0)

# ── 3. Upload submission ─────────────────────────────────────────────────
print("Uploading submission...")
resp = requests.post(
    f"{API_URL}/submissions",
    headers={"X-Team-Token": TEAM_TOKEN, "Host": "eval.yourdomain.com"},
    files={"file": ("submission.zip", buf, "application/zip")},
    data={"team_id": TEAM_ID},
)
if resp.status_code != 202:
    raise SystemExit(f"Upload failed ({resp.status_code}): {resp.text}")

submission_id = resp.json()["submission_id"]
print(f"✓ Submitted. ID: {submission_id}")

# ── 4. Poll until evaluation completes ───────────────────────────────────
print("Waiting for evaluation (build + run, typically 2–5 min)...")
while True:
    s = requests.get(
        f"{API_URL}/submissions/{submission_id}",
        headers={"X-Team-Token": TEAM_TOKEN, "Host": "eval.yourdomain.com"},
    ).json()
    status = s["status"]
    print(f"  [{status}] {s.get('status_message') or ''}")
    if status in ("COMPLETED", "EVAL_FAILED", "BUILD_FAILED", "TIMEOUT"):
        break
    time.sleep(10)

# ── 5. Fetch the score ──────────────────────────────────────────────────
if status == "COMPLETED":
    score = requests.get(
        f"{API_URL}/submissions/{submission_id}/score",
        headers={"X-Team-Token": TEAM_TOKEN, "Host": "eval.yourdomain.com"},
    ).json()
    print("\n✅ Submission scored!")
    print(json.dumps(score, indent=2))
else:
    print(f"\n❌ Submission ended with status: {status}")
    logs = requests.get(
        f"{API_URL}/submissions/{submission_id}/logs",
        headers={"X-Team-Token": TEAM_TOKEN, "Host": "eval.yourdomain.com"},
    ).text
    print("Last 2000 chars of logs:\n", logs[-2000:])


Uploading submission...
✓ Submitted. ID: 1b29b523-105e-4651-a1fb-de13ee2bf1c0
Waiting for evaluation (build + run, typically 2–5 min)...
  [BUILDING] 
  [BUILDING] 
  [BUILDING] 
  [BUILDING] 
  [BUILDING] 
  [BUILDING] 
  [EVALUATING] 
  [EVALUATING] 
  [EVALUATING] 
  [COMPLETED] 

✅ Submission scored!
{
  "submission_id": "1b29b523-105e-4651-a1fb-de13ee2bf1c0",
  "team_name": "Playtest Team 01",
  "final_score": 1077.8418157573035,
  "cost_breakdown": {
    "fcas_revenue": 0.0,
    "export_tariff": 0.0,
    "import_tariff": 0.0,
    "carbon_cost": 92.60612826214178,
    "diesel_fuel_cost": 0.0,
    "ramp_charge": 154.38214987522596,
    "blackout_penalty": 0.0,
    "overvoltage_charge": 0.0,
    "battery_wear": 27.10367947221212,
    "peak_demand_charge": 0.0
  },
  "component_scores": {
    "cost": null,
    "renewable": null,
    "stability": null,
    "reliability": null
  },
  "renewable_ratio": 0.3946449597419127,
  "scored_at": "2026-05-24 07:43:41.087458+00:00",
  "scenarios"

## 7. Agentic Hybrid Controller (OpenAI)

For agentic scenarios, you can define a `Strategy` class with `.plan()` and `.replan()` methods to use an LLM. The LLM can read the scenario briefing and set a high-level strategy (like a target SOC), which the fast `.step()` method then executes.

**Important:** When submitting to the judging server, your pod will not have raw internet access. You must use the internal LLM proxy by passing ase_url=os.environ.get('OPENAI_BASE_URL') to the OpenAI client, and use a dummy API key if the real one isn't present.

In [ ]:
import os
# If you're using an LLM, you would normally import the client here.
# import openai

class Strategy:
    def __init__(self):
        # The environment will inject OPENAI_API_KEY and OPENAI_BASE_URL.
        # We MUST use OPENAI_BASE_URL to point to the internal proxy!
        self.base_url = os.environ.get('OPENAI_BASE_URL')
        self.api_key = os.environ.get('OPENAI_API_KEY', 'dummy-key')
        # self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)
        self.target_soc = 0.5

    def plan(self, state):
        # Called once at the beginning of an agentic scenario.
        # State contains scenario briefing text.
        briefing = state.get('qualitative_briefing', '')

        # Example: Call LLM to parse strategy
        # response = self.client.chat.completions.create(...)
        # self.target_soc = float(response...)

        return {'agent_plan': {'target_soc': self.target_soc}}

    def replan(self, state, alerts):
        # Called when mid-run qualitative alerts occur.
        return {'agent_plan': {'target_soc': self.target_soc}}

    def step(self, state):
        # Called every simulation step. Fast, no LLM calls!
        plan = state.get('agent_plan', {})
        target = plan.get('target_soc', self.target_soc)
        current_soc = state.get('soc', 0.0)

        flow = 0.0
        if current_soc < target:
            flow = -20.0 # charge
        elif current_soc > target:
            flow = 20.0 # discharge

        return {
            'battery_flow_kw': flow,
            'emergency_generator': 0.0,
            'curtail_solar': 0.0,
            'fcas_reserve_kw': 0.0
        }
